In [1]:
# ==============================================================================
# CELL 0: OPUS CLOUD KERNEL (Codespaces & Service Account)
# ==============================================================================
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from google.cloud import bigquery
from google.oauth2 import service_account

# --- 1. CLOUD CONFIGURATION & PATHS ---
PROJECT_ID = '645009831643'   # Project ID real alineado con tu SA
DATASET_REAL = 'pienza_mini'  # Star Schema (Realidad)
DATASET_SYNTH = 'pienza_big'  # Manifold (Simulación)

SA_PATH = "/workspaces/pienza/secrets/service-account.json"
DUMP_DIR = "/workspaces/pienza/data/dumped_files/"

# --- 2. AUTHENTICATION & CLIENT INITIALIZATION ---
print("🔐 Iniciando Solicitud de Soberanía Cloud (Autenticación via SA)...")
if os.path.exists(SA_PATH):
    credentials = service_account.Credentials.from_service_account_file(SA_PATH)
    client = bigquery.Client(credentials=credentials, project=PROJECT_ID)
    print(f"🚀 Cliente BigQuery Activo en proyecto: {PROJECT_ID}")
else:
    print(f"🔴 ERROR CRÍTICO: No se encontró el Service Account en {SA_PATH}")
    raise FileNotFoundError("Service Account missing.")

# --- 3. VISUAL CANON (OPUS LAB THEME) ---
OPUS_PURPLE, OPUS_TEAL, OPUS_GREY, OPUS_TEXT = '#440154', '#21918c', '#FAFAFA', '#121212'

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'figure.facecolor': OPUS_GREY, 'axes.facecolor': OPUS_GREY, 'text.color': OPUS_TEXT,
    'axes.titlecolor': OPUS_PURPLE, 'axes.titleweight': 'bold', 'figure.titlesize': 20
})

print("✅ Visual Identity Loaded: Opus Lab (Cloud Mode).")
print(f"🔗 Ready to operate on {DATASET_REAL} (Reality) and {DATASET_SYNTH} (Simulation).")

🔐 Iniciando Solicitud de Soberanía Cloud (Autenticación via SA)...
🚀 Cliente BigQuery Activo en proyecto: 645009831643
✅ Visual Identity Loaded: Opus Lab (Cloud Mode).
🔗 Ready to operate on pienza_mini (Reality) and pienza_big (Simulation).


In [2]:
# ==============================================================================
# CELL 1: GCS STAGING & EXTERNAL TABLE FORGE
# ==============================================================================
# Purpose: 1. Ensure the Parquet exists in GCS Staging.
#          2. Create an External Table in BigQuery (Zero BQ Storage Cost).
# Logic:   Idempotent Upload -> Dataset Check -> External Table Mapping.
# ==============================================================================
from google.cloud import storage, bigquery
import os

# --- 1. CONFIGURACIÓN DE RUTAS ---
BUCKET_NAME = 'pienza-streamlit'
LOCAL_PATH  = '/workspaces/pienza/data/dumped_files/df_gan_training_set_v8.parquet'
GCS_PATH    = 'training/df_gan_training_set_v8.parquet'

DEST_DATASET = 'pienza_big'
DEST_TABLE   = 'gan_training_set_v8_reference'
TABLE_ID     = f"{PROJECT_ID}.{DEST_DATASET}.{DEST_TABLE}"

# --- 2. PROTOCOLO DE STAGING (GCS) ---
def sync_to_staging():
    print(f"📡 Verificando Staging en GCS: {BUCKET_NAME}...")
    storage_client = storage.Client(credentials=credentials, project=PROJECT_ID)
    bucket = storage_client.bucket(BUCKET_NAME)
    blob = bucket.blob(GCS_PATH)

    if blob.exists():
        print(f"   ✅ OBJETO DETECTADO: '{GCS_PATH}' ya está en la nube.")
    else:
        print(f"   ⚠️  OBJETO FALTANTE: Iniciando ascenso de {os.path.basename(LOCAL_PATH)}...")
        blob.upload_from_filename(LOCAL_PATH)
        print(f"   🚀 ASCENSO EXITOSO: gs://{BUCKET_NAME}/{GCS_PATH}")

# --- 3. ASEGURAR DATASET (BQ) ---
def ensure_dataset_exists():
    dataset_ref = client.dataset(DEST_DATASET)
    try:
        client.get_dataset(dataset_ref)
        print(f"✅ DATASET CONFIRMADO: '{DEST_DATASET}' listo.")
    except Exception:
        print(f"🏗️  DATASET FALTANTE: Creando '{DEST_DATASET}'...")
        dataset = bigquery.Dataset(dataset_ref)
        dataset.location = "US"
        client.create_dataset(dataset)
        print(f"   ✅ Dataset creado exitosamente.")

# --- 4. FORJAR TABLA EXTERNA (BQ) ---
def create_external_table():
    print(f"🏗️  Forjando enlace externo en BigQuery...")
    
    source_uri = f"gs://{BUCKET_NAME}/{GCS_PATH}"

    # Configuración de Tabla Externa (PARQUET)
    external_config = bigquery.ExternalConfig("PARQUET")
    external_config.source_uris = [source_uri]
    external_config.autodetect = True  # BQ infiere el esquema del Parquet automáticamente

    # Crear/Reemplazar la tabla
    table = bigquery.Table(TABLE_ID)
    table.external_data_configuration = external_config

    client.delete_table(TABLE_ID, not_found_ok=True)
    client.create_table(table)

    print(f"🔗 ENLACE EXITOSO: '{DEST_TABLE}' activo en '{DEST_DATASET}'.")
    print(f"💰 Costo de almacenamiento BQ: $0.00 (Sovereign GCS Storage).")

# --- EJECUCIÓN DEL FLUJO ---
print("🚀 INICIANDO PIPELINE DE INGESTA GENERATIVA...")
print("-" * 65)
sync_to_staging()
ensure_dataset_exists()
create_external_table()
print("-" * 65)
print("🏆 ESTATUS: EL TRAINING SET V8 ESTÁ EN LÍNEA.")

🚀 INICIANDO PIPELINE DE INGESTA GENERATIVA...
-----------------------------------------------------------------
📡 Verificando Staging en GCS: pienza-streamlit...
   ⚠️  OBJETO FALTANTE: Iniciando ascenso de df_gan_training_set_v8.parquet...
   🚀 ASCENSO EXITOSO: gs://pienza-streamlit/training/df_gan_training_set_v8.parquet
✅ DATASET CONFIRMADO: 'pienza_big' listo.
🏗️  Forjando enlace externo en BigQuery...
🔗 ENLACE EXITOSO: 'gan_training_set_v8_reference' activo en 'pienza_big'.
💰 Costo de almacenamiento BQ: $0.00 (Sovereign GCS Storage).
-----------------------------------------------------------------
🏆 ESTATUS: EL TRAINING SET V8 ESTÁ EN LÍNEA.


In [3]:
# ==============================================================================
# CELL 2: DESPLIEGUE DEL MANIFOLD V8 (SIMULACIÓN)
# ==============================================================================
# Purpose: Conecta el Manifold V8 (Canon) en GCS a BigQuery.
# Logic:   GCS Staging Check -> External Table Forge (Zero Cost).
# ==============================================================================
from google.cloud import storage, bigquery
import os

# --- 1. CONFIGURACIÓN DEL UNIVERSO V8 ---
MANIFOLD_FILENAME = '260426_cGAN_manifold_v8.parquet'
LOCAL_MANIFOLD_PATH = f'/workspaces/pienza/data/dumped_files/{MANIFOLD_FILENAME}'
GCS_MANIFOLD_PATH = f'manifold/{MANIFOLD_FILENAME}'

TABLE_NAME_MANIFOLD = 'synthetic_manifold_v8'
MANIFOLD_TABLE_ID = f"{PROJECT_ID}.{DATASET_SYNTH}.{TABLE_NAME_MANIFOLD}"

def deploy_synthetic_universe_v8():
    print(f"🌉 Iniciando conexión del Manifold Canon V8 a {DATASET_SYNTH}...")
    print("-" * 65)

    try:
        # A. STAGING: Asegurar que el Manifold esté en GCS
        storage_client = storage.Client(credentials=credentials, project=PROJECT_ID)
        bucket = storage_client.bucket(BUCKET_NAME)
        blob = bucket.blob(GCS_MANIFOLD_PATH)

        if blob.exists():
            print(f"   ✅ CANON DETECTADO: '{GCS_MANIFOLD_PATH}' ya reside en GCS.")
        else:
            if os.path.exists(LOCAL_MANIFOLD_PATH):
                print(f"   ⚠️  CANON FALTANTE: Subiendo Manifold V8 desde local...")
                blob.upload_from_filename(LOCAL_MANIFOLD_PATH)
                print(f"   🚀 ASCENSO EXITOSO: gs://{BUCKET_NAME}/{GCS_MANIFOLD_PATH}")
            else:
                raise FileNotFoundError(f"❌ Error Crítico: No se encontró el archivo local en {LOCAL_MANIFOLD_PATH}")

        # B. FORJA: Configurar Tabla Externa en BigQuery
        print(f"   🏗️  Forjando tabla externa: {TABLE_NAME_MANIFOLD}...")
        
        uri = f"gs://{BUCKET_NAME}/{GCS_MANIFOLD_PATH}"
        ext_config = bigquery.ExternalConfig("PARQUET")
        ext_config.source_uris = [uri]
        ext_config.autodetect = True

        table = bigquery.Table(MANIFOLD_TABLE_ID)
        table.external_data_configuration = ext_config

        # C. DESPLIEGUE: Reemplazar tabla para asegurar frescura del puntero
        client.delete_table(MANIFOLD_TABLE_ID, not_found_ok=True)
        client.create_table(table)

        print(f"   ✅ MANIFOLD V8 DESPLEGADO.")
        print(f"   🔗 Tabla Cloud: {MANIFOLD_TABLE_ID}")
        print(f"   💰 Costo de Almacenamiento BQ: $0.00")

    except Exception as e:
        print(f"   ❌ ERROR CRÍTICO EN EL DESPLIEGUE: {e}")

# EJECUTAR EL DESPLIEGUE
deploy_synthetic_universe_v8()

# TEST RÁPIDO DE SOBERANÍA
try:
    query = f"SELECT COUNT(*) as total FROM `{MANIFOLD_TABLE_ID}`"
    count = client.query(query).to_dataframe().iloc[0]['total']
    print(f"\n📊 AUDITORÍA: {count:,} filas detectadas en el Manifold V8.")
except Exception as e:
    print(f"⚠️ Error en auditoría: {e}")

🌉 Iniciando conexión del Manifold Canon V8 a pienza_big...
-----------------------------------------------------------------
   ⚠️  CANON FALTANTE: Subiendo Manifold V8 desde local...
   🚀 ASCENSO EXITOSO: gs://pienza-streamlit/manifold/260426_cGAN_manifold_v8.parquet
   🏗️  Forjando tabla externa: synthetic_manifold_v8...
   ✅ MANIFOLD V8 DESPLEGADO.
   🔗 Tabla Cloud: 645009831643.pienza_big.synthetic_manifold_v8
   💰 Costo de Almacenamiento BQ: $0.00

📊 AUDITORÍA: 1,010,001 filas detectadas en el Manifold V8.


In [4]:
# TEST RÁPIDO DE SOBERANÍA DE DATOS
for table_name in ['gan_training_set_v8_reference', 'synthetic_manifold_v8']:
    query = f"SELECT COUNT(*) as total FROM `{PROJECT_ID}.{DATASET_SYNTH}.{table_name}`"
    count = client.query(query).to_dataframe().iloc[0]['total']
    print(f"✅ Tabla {table_name}: {count} filas detectadas.")

✅ Tabla gan_training_set_v8_reference: 5123 filas detectadas.
✅ Tabla synthetic_manifold_v8: 1010001 filas detectadas.
